# Datasplit Overview

This notebook summarizes the monthly distribution of `Data/dataset_construction/null_signal_dataset.parquet`.

Current scope:
- group by publish date month
- treat `label = 0` as `null_count`
- show monthly totals, label counts, ratios, and key missing-value summaries
- auto-adapt to renamed columns (new schema first, legacy schema fallback)

In [1]:
from pathlib import Path

import pandas as pd
from IPython.display import display

def resolve_data_path() -> Path:
    candidates = [
        Path('./Data/dataset_construction/null_signal_dataset.parquet'),
        Path('../Data/dataset_construction/null_signal_dataset.parquet'),
        Path('./dataset_construction/null_signal_dataset.parquet'),
    ]
    for candidate in candidates:
        if candidate.exists():
            return candidate.resolve()
    raise FileNotFoundError(
        'Could not find null_signal_dataset.parquet. Tried: '
        + ', '.join(str(path) for path in candidates)
    )

DATA_PATH = resolve_data_path()
df = pd.read_parquet(DATA_PATH)

print(f'Data path: {DATA_PATH}')
print(f'Rows: {len(df):,}')
print(f'Columns: {len(df.columns)}')
print('\nDtypes:')
display(df.dtypes.rename('dtype').to_frame())

print('\nFirst 5 rows:')
display(df.head())

print('\nLabel distribution:')
display(df['label'].value_counts(dropna=False).rename_axis('label').reset_index(name='count').sort_values('label'))

Data path: C:\Users\纯情小火鸡\Desktop\bilby2\bilby2\Bilby code + output\code\Model\gta_multi_source_6_combine_dataset.xlsx
Rows: 1,255
Columns: 9

Dtypes:


,dtype
title,str
title_normalized,str
source_url,str
site_root_url,str
title_en,str
summary,str
published_date,datetime64[us]
type,str
label,int64



First 5 rows:


,title,title_normalized,source_url,site_root_url,title_en,summary,published_date,type,label
0,财政部迅速下达4.84亿元救灾资金 支持多地秋粮抢收抢烘 积极应对连阴雨,财政部迅速下达484亿元救灾资金支持多地秋粮抢收抢烘积极应对连阴雨,http://nys.mof.gov.cn/bgtGongZuoDongTai_1_1_1_...,nys.mof.gov.cn,The Ministry of Finance quickly allocated 484 ...,2025年10月11日，财政部与农业农村部下达4.84亿元中央财政资金，支持7省农业生产防灾...,2025-10-11,bilby,1
1,国务院关税税则委员会关于停止实施对原产于美国的部分进口商品加征关税措施的公告,国务院关税税则委员会关于停止实施对原产于美国的部分进口商品加征关税措施的公告,http://gss.mof.gov.cn/gzdt/zhengcefabu/202511/...,gss.mof.gov.cn,Announcement of the Customs Tariff Commission ...,2025年11月10日起，中国停止对原产于美国的部分进口商品加征关税，以落实中美经贸磋商成果共识。,2025-11-05,bilby,1
2,国务院关税税则委员会关于调整对原产于美国的进口商品加征关税措施的公告,国务院关税税则委员会关于调整对原产于美国的进口商品加征关税措施的公告,http://gss.mof.gov.cn/gzdt/zhengcefabu/202511/...,gss.mof.gov.cn,Announcement of the State Council Customs Tari...,2025年11月10日起，中国将暂停实施对原产于美国的进口商品24%的加征关税税率，保留10...,2025-11-05,bilby,1
3,关于“十五五”期间支持科技创新进口税收优惠政策的通知,关于十五五期间支持科技创新进口税收优惠政策的通知,http://gss.mof.gov.cn/gzdt/zhengcefabu/202602/...,gss.mof.gov.cn,Notice on Preferential Import Tax Policies for...,为支持科技创新，“十五五”期间（2026-2030年）科研、教学等机构进口特定用品将免征关税...,2026-02-14,bilby,1
4,国务院关税税则委员会公布公告停止实施对原产于美国的部分进口商品加征关税措施,国务院关税税则委员会公布公告停止实施对原产于美国的部分进口商品加征关税措施,http://gss.mof.gov.cn/gzdt/zhengcejiedu/202511...,gss.mof.gov.cn,The State Council Customs Tariff Commission an...,2025年11月5日，国务院关税税则委员会宣布停止对原产于美国的部分进口商品加征关税，以落实...,2025-11-05,bilby,1



Label distribution:


,label,count
0,0,1153
1,1,102


In [2]:
analysis_df = df.copy()

required_columns = [
    'title',
    'summary',
    'title_normalized',
    'source_url',
    'site_root_url',
    'title_en',
    'published_date',
    'type',
    'label',
]
missing_required = [column for column in required_columns if column not in analysis_df.columns]
if missing_required:
    raise KeyError(f"Missing required columns in input dataset: {missing_required}")

DATE_COL = 'published_date'
ROOT_COL = 'site_root_url'
URL_COL = 'source_url'
TITLE_COL = 'title'

analysis_df[DATE_COL] = pd.to_datetime(analysis_df[DATE_COL], errors='coerce')
analysis_df['month'] = analysis_df[DATE_COL].dt.to_period('M').astype('string')
analysis_df['month'] = analysis_df['month'].fillna('MISSING')
analysis_df[ROOT_COL] = analysis_df[ROOT_COL].fillna('MISSING')
analysis_df['type'] = analysis_df['type'].fillna('MISSING')

unexpected_labels = analysis_df.loc[~analysis_df['label'].isin([0, 1]), 'label']
print(f'Unexpected label rows: {len(unexpected_labels):,}')
if not unexpected_labels.empty:
    display(unexpected_labels.value_counts(dropna=False).rename_axis('label').reset_index(name='count'))

monthly_distribution = (
    analysis_df.groupby('month', dropna=False)
    .agg(
        total_rows=('label', 'size'),
        label_1_count=('label', lambda values: int((values == 1).sum())),
        label_0_count=('label', lambda values: int((values == 0).sum())),
    )
    .reset_index()
    .sort_values('month')
)
monthly_distribution['label_1_ratio'] = (
    monthly_distribution['label_1_count'] / monthly_distribution['total_rows']
).round(2)
monthly_distribution['label_0_ratio'] = (
    monthly_distribution['label_0_count'] / monthly_distribution['total_rows']
).round(2)

source_root_counts = (
    analysis_df.groupby(['month', ROOT_COL], dropna=False)
    .size()
    .unstack(fill_value=0)
    .reset_index()
    .sort_values('month')
)

monthly_distribution = monthly_distribution.merge(source_root_counts, on='month', how='left')

base_columns = {'month', 'total_rows', 'label_1_count', 'label_0_count', 'label_1_ratio', 'label_0_ratio'}
root_count_columns = [column for column in monthly_distribution.columns if column not in base_columns]

print('Resolved columns:')
print(
    {
        'date': DATE_COL,
        'root_url': ROOT_COL,
        'source_url': URL_COL,
        'title': TITLE_COL,
        'summary': 'summary',
        'title_normalized': 'title_normalized',
        'title_en': 'title_en',
        'type': 'type',
    }
)

print('Monthly distribution summary:')
display(monthly_distribution)

print('Verification checks:')
print('Total rows match:', int(monthly_distribution['total_rows'].sum()) == len(analysis_df))
print(
    'Monthly label counts match total:',
    bool(((monthly_distribution['label_1_count'] + monthly_distribution['label_0_count']) == monthly_distribution['total_rows']).all()),
)
print(
    'Per-month source root counts match total:',
    bool((monthly_distribution[root_count_columns].sum(axis=1) == monthly_distribution['total_rows']).all()),
)

Unexpected label rows: 0
Resolved columns:
{'date': 'published_date', 'root_url': 'site_root_url', 'source_url': 'source_url', 'title': 'title', 'summary': 'summary', 'title_normalized': 'title_normalized', 'title_en': 'title_en', 'type': 'type'}
Monthly distribution summary:


,month,total_rows,label_1_count,label_0_count,label_1_ratio,label_0_ratio,gss.mof.gov.cn,jjs.mof.gov.cn,jrs.mof.gov.cn,nys.mof.gov.cn,www.cdb.com.cn,www.mofcom.gov.cn
0,2025-06,81,2,79,0.02,0.98,0,0,1,1,20,59
1,2025-07,116,6,110,0.05,0.95,3,0,1,1,16,95
2,2025-08,96,11,85,0.11,0.89,2,0,5,2,14,73
3,2025-09,172,21,151,0.12,0.88,0,4,0,1,16,151
4,2025-10,183,19,164,0.10,0.90,4,0,0,1,22,156
5,2025-11,165,15,150,0.09,0.91,4,0,1,0,17,143
6,2025-12,212,9,203,0.04,0.96,3,0,3,0,10,196
7,2026-01,120,13,107,0.11,0.89,5,2,5,0,18,90
8,2026-02,110,6,104,0.05,0.95,9,1,0,1,17,82


Verification checks:
Total rows match: True
Monthly label counts match total: True
Per-month source root counts match total: True


In [3]:
key_columns = [
    URL_COL,
    ROOT_COL,
    DATE_COL,
    'title',
    'summary',
    'title_normalized',
    'title_en',
    'type',
    'label',
]
if 'data_type' in analysis_df.columns:
    key_columns.append('data_type')

missing_overall = pd.DataFrame(
    {
        'column': key_columns,
        'missing_count': [int(analysis_df[column].isna().sum()) for column in key_columns],
        'missing_ratio': [analysis_df[column].isna().mean() for column in key_columns],
    }
).sort_values(['missing_count', 'column'], ascending=[False, True])

missing_by_month = (
    analysis_df.groupby('month', dropna=False)[key_columns]
    .apply(lambda group: group.isna().sum())
    .reset_index()
    .sort_values('month')
)

print('Overall missing summary for key columns:')
display(missing_overall)

print('Missing summary by month for key columns:')
display(missing_by_month)

Overall missing summary for key columns:


,column,missing_count,missing_ratio
5,title_normalized,41,0.032669
6,title_en,19,0.015139
8,label,0,0.000000
2,published_date,0,0.000000
1,site_root_url,0,0.000000
0,source_url,0,0.000000
4,summary,0,0.000000
3,title,0,0.000000
7,type,0,0.000000


Missing summary by month for key columns:


,month,source_url,site_root_url,published_date,title,summary,title_normalized,title_en,type,label
0,2025-06,0,0,0,0,0,2,0,0,0
1,2025-07,0,0,0,0,0,2,1,0,0
2,2025-08,0,0,0,0,0,3,0,0,0
3,2025-09,0,0,0,0,0,10,4,0,0
4,2025-10,0,0,0,0,0,5,7,0,0
5,2025-11,0,0,0,0,0,5,3,0,0
6,2025-12,0,0,0,0,0,6,1,0,0
7,2026-01,0,0,0,0,0,6,1,0,0
8,2026-02,0,0,0,0,0,2,2,0,0


## Dataset Split (Expanding Window)

Current split rule for this notebook (time-dependent):
- method: monthly-step expanding window
- folds: 4
- train window: 120 days
- valid window: 30 days
- test window: 30 days (folds 0-2), variable (fold 3: max data date)
- fold start date: dataset minimum published date
- train in later folds includes prior valid/test ranges

In [4]:
split_base_df = analysis_df.copy()

split_base_df[DATE_COL] = pd.to_datetime(split_base_df[DATE_COL], errors='coerce')
date_valid_mask = split_base_df[DATE_COL].notna()
if not date_valid_mask.any():
    raise ValueError('No valid published_date values are available for expanding-window split.')

split_base_df = split_base_df.loc[date_valid_mask].copy()

FOLD_CONFIG = {
    'window_granularity': 'month',
    'n_folds': 4,
    'train_days': 120,
    'valid_days': 30,
    'test_days': 30,
    'start_date': split_base_df[DATE_COL].min().normalize(),
    'expand_rule': 'include_previous_valid_test',
}

if FOLD_CONFIG['window_granularity'] != 'month':
    raise ValueError("This notebook currently implements monthly-step expanding windows only.")

split_frames = []
fold_definitions = []
max_date = split_base_df[DATE_COL].max()

for fold_id in range(FOLD_CONFIG['n_folds']):
    fold_anchor = (FOLD_CONFIG['start_date'] + pd.DateOffset(months=fold_id)).normalize()
    train_start = FOLD_CONFIG['start_date']
    train_end = (fold_anchor + pd.Timedelta(days=FOLD_CONFIG['train_days'] - 1)).replace(hour=23, minute=59, second=59)
    valid_start = train_end + pd.Timedelta(seconds=1)
    valid_end = (valid_start + pd.Timedelta(days=FOLD_CONFIG['valid_days'] - 1)).replace(hour=23, minute=59, second=59)
    test_start = valid_end + pd.Timedelta(seconds=1)

    # Keep 30-day test for folds 0-2; extend last fold test to max data date.
    if fold_id == FOLD_CONFIG['n_folds'] - 1:
        test_end = max_date.replace(hour=23, minute=59, second=59)
    else:
        test_end = (test_start + pd.Timedelta(days=FOLD_CONFIG['test_days'] - 1)).replace(hour=23, minute=59, second=59)

    fold_scope_mask = split_base_df[DATE_COL].between(train_start, test_end, inclusive='both')
    fold_df = split_base_df.loc[fold_scope_mask].copy()
    fold_df['fold_id'] = fold_id
    fold_df['dataset'] = 'out_of_window'
    fold_df.loc[fold_df[DATE_COL].between(train_start, train_end, inclusive='both'), 'dataset'] = 'train'
    fold_df.loc[fold_df[DATE_COL].between(valid_start, valid_end, inclusive='both'), 'dataset'] = 'valid'
    fold_df.loc[fold_df[DATE_COL].between(test_start, test_end, inclusive='both'), 'dataset'] = 'test'
    fold_df = fold_df.loc[fold_df['dataset'].isin(['train', 'valid', 'test'])].copy()

    split_frames.append(fold_df)
    fold_definitions.append(
        {
            'fold_id': fold_id,
            'train_start': train_start,
            'train_end': train_end,
            'valid_start': valid_start,
            'valid_end': valid_end,
            'test_start': test_start,
            'test_end': test_end,
            'max_data_date': max_date,
            'rows_in_window': len(fold_df),
            'train_rows': int((fold_df['dataset'] == 'train').sum()),
            'valid_rows': int((fold_df['dataset'] == 'valid').sum()),
            'test_rows': int((fold_df['dataset'] == 'test').sum()),
            'label_1_rows': int((fold_df['label'] == 1).sum()),
            'label_0_rows': int((fold_df['label'] == 0).sum()),
            'has_train': bool((fold_df['dataset'] == 'train').any()),
            'has_valid': bool((fold_df['dataset'] == 'valid').any()),
            'has_test': bool((fold_df['dataset'] == 'test').any()),
            'temporal_order_ok': bool(train_end < valid_start and valid_end < test_start),
        }
    )

split_df = pd.concat(split_frames, ignore_index=True) if split_frames else pd.DataFrame(columns=list(split_base_df.columns) + ['fold_id', 'dataset'])
fold_definition_df = pd.DataFrame(fold_definitions).sort_values('fold_id').reset_index(drop=True)

split_summary = (
    split_df.groupby(['fold_id', 'dataset'], dropna=False)
    .agg(
        total_rows=('label', 'size'),
        label_1_count=('label', lambda values: int((values == 1).sum())),
        label_0_count=('label', lambda values: int((values == 0).sum())),
    )
    .reset_index()
    .sort_values(['fold_id', 'dataset'], key=lambda series: series.map({'train': 0, 'valid': 1, 'test': 2}).fillna(series))
)
split_summary['label_1_ratio'] = (split_summary['label_1_count'] / split_summary['total_rows']).round(4)
split_summary['label_0_ratio'] = (split_summary['label_0_count'] / split_summary['total_rows']).round(4)

source_distribution = (
    split_df.groupby(['fold_id', ROOT_COL, 'dataset'], dropna=False)
    .size()
    .rename('count')
    .reset_index()
    .pivot_table(index=['fold_id', ROOT_COL], columns='dataset', values='count', fill_value=0)
    .reset_index()
    .rename(columns={ROOT_COL: 'source_type'})
)
for split_name in ['train', 'valid', 'test']:
    if split_name not in source_distribution.columns:
        source_distribution[split_name] = 0
source_distribution = source_distribution[['fold_id', 'source_type', 'train', 'valid', 'test']].sort_values(['fold_id', 'source_type']).reset_index(drop=True)

validation_checks = pd.DataFrame(
    [
        {
            'check': 'at_least_one_fold',
            'passed': bool(len(fold_definition_df) > 0),
        },
        {
            'check': 'fold_temporal_order_ok',
            'passed': bool(fold_definition_df['temporal_order_ok'].all()) if not fold_definition_df.empty else False,
        },
        {
            'check': 'every_fold_has_train',
            'passed': bool(fold_definition_df['has_train'].all()) if not fold_definition_df.empty else False,
        },
        {
            'check': 'every_fold_has_valid',
            'passed': bool(fold_definition_df['has_valid'].all()) if not fold_definition_df.empty else False,
        },
        {
            'check': 'every_fold_has_test',
            'passed': bool(fold_definition_df['has_test'].all()) if not fold_definition_df.empty else False,
        },
        {
            'check': 'all_rows_have_fold_id',
            'passed': bool(split_df['fold_id'].notna().all()) if not split_df.empty else False,
        },
        {
            'check': 'all_rows_have_dataset',
            'passed': bool(split_df['dataset'].isin(['train', 'valid', 'test']).all()) if not split_df.empty else False,
        },
    ]
)

print('Expanding-window configuration:')
display(pd.DataFrame([FOLD_CONFIG]))

print('Fold definitions:')
display(fold_definition_df)

last_fold = fold_definition_df.loc[fold_definition_df['fold_id'].eq(FOLD_CONFIG['n_folds'] - 1)]
if not last_fold.empty:
    last_row = last_fold.iloc[0]
    print(
        f"Last fold test range: {last_row['test_start'].date()} -> {last_row['test_end'].date()}"
    )

print('Split summary by fold and dataset:')
display(split_summary)

print('Source distribution by fold/site_root_url:')
display(source_distribution.head(30))

print('Validation checks:')
display(validation_checks)
print('Overall split rows:', len(split_df))

Expanding-window configuration:


,window_granularity,n_folds,train_days,valid_days,test_days,start_date,expand_rule
0,month,4,120,30,30,2025-06-03,include_previous_valid_test


Fold definitions:


,fold_id,train_start,train_end,valid_start,valid_end,test_start,test_end,max_data_date,rows_in_window,train_rows,valid_rows,test_rows,label_1_rows,label_0_rows,has_train,has_valid,has_test,temporal_order_ok
0,0,2025-06-03,2025-09-30 23:59:59,2025-10-01,2025-10-30 23:59:59,2025-10-31,2025-11-29 23:59:59,2026-02-28,811,465,178,168,74,737,True,True,True,True
1,1,2025-06-03,2025-10-30 23:59:59,2025-10-31,2025-11-29 23:59:59,2025-11-30,2025-12-29 23:59:59,2026-02-28,972,643,168,161,82,890,True,True,True,True
2,2,2025-06-03,2025-11-30 23:59:59,2025-12-01,2025-12-30 23:59:59,2025-12-31,2026-01-29 23:59:59,2026-02-28,1136,813,178,145,95,1041,True,True,True,True
3,3,2025-06-03,2025-12-31 23:59:59,2026-01-01,2026-01-30 23:59:59,2026-01-31,2026-02-28 23:59:59,2026-02-28,1255,1025,119,111,102,1153,True,True,True,True


Last fold test range: 2026-01-31 -> 2026-02-28
Split summary by fold and dataset:


...

Source distribution by fold/site_root_url:


dataset,fold_id,source_type,train,valid,test
0,0,gss.mof.gov.cn,5.0,4.0,4.0
1,0,jjs.mof.gov.cn,4.0,0.0,0.0
2,0,jrs.mof.gov.cn,7.0,0.0,1.0
3,0,nys.mof.gov.cn,5.0,1.0,0.0
4,0,www.cdb.com.cn,66.0,22.0,17.0
5,0,www.mofcom.gov.cn,378.0,151.0,146.0
6,1,gss.mof.gov.cn,9.0,4.0,2.0
7,1,jjs.mof.gov.cn,4.0,0.0,0.0
8,1,jrs.mof.gov.cn,7.0,1.0,3.0
9,1,nys.mof.gov.cn,6.0,0.0,0.0


Validation checks:


,check,passed
0,at_least_one_fold,True
1,fold_temporal_order_ok,True
2,every_fold_has_train,True
3,every_fold_has_valid,True
4,every_fold_has_test,True
5,all_rows_have_fold_id,True
6,all_rows_have_dataset,True


Overall split rows: 4174


In [5]:
export_columns = {
    'fold_id': 'fold_id',
    'dataset': 'dataset',
    DATE_COL: 'date',
    ROOT_COL: 'root_url',
    URL_COL: 'source_url',
    'title': 'title',
    'summary': 'summary',
    'title_normalized': 'title_normalized',
    'title_en': 'title_en',
    'type': 'type',
    'label': 'label',
}

if 'data_type' in split_df.columns:
    export_columns['data_type'] = 'data_type'

split_output_dir = Path('./Data/dataset_construction').resolve()
split_output_dir.mkdir(parents=True, exist_ok=True)
combined_output_path = split_output_dir / 'ewsplit_dataset.parquet'

sort_columns = ['fold_id', 'dataset', 'date', 'root_url', 'source_url']
if 'type' in export_columns.values():
    sort_columns.append('type')
if 'data_type' in export_columns.values():
    sort_columns.append('data_type')

export_df = (
    split_df.loc[:, list(export_columns.keys())]
    .rename(columns=export_columns)
    .sort_values(sort_columns)
    .reset_index(drop=True)
)

title_series = export_df['title'].fillna('').astype(str)
has_zh = title_series.str.contains('[一-龥]', regex=True)
has_en = title_series.str.contains('[A-Za-z]', regex=True)
export_df['title_lang'] = 'zh'
export_df.loc[has_en & ~has_zh, 'title_lang'] = 'en'

export_df.to_parquet(combined_output_path, index=False)

export_summary_df = pd.DataFrame(
    [
        {
            'output_path': str(combined_output_path),
            'rows_written': len(export_df),
            'fold_count': int(export_df['fold_id'].nunique()) if not export_df.empty else 0,
            'datasets': ', '.join(sorted(export_df['dataset'].dropna().unique().tolist())),
        }
    ]
)

export_distribution = (
    export_df.groupby(['fold_id', 'dataset'], dropna=False)
    .agg(
        total_rows=('label', 'size'),
        label_1_count=('label', lambda values: int((values == 1).sum())),
        label_0_count=('label', lambda values: int((values == 0).sum())),
        title_zh_count=('title_lang', lambda values: int((values == 'zh').sum())),
        title_en_count=('title_lang', lambda values: int((values == 'en').sum())),
    )
    .reset_index()
    .sort_values(['fold_id', 'dataset'], key=lambda series: series.map({'train': 0, 'valid': 1, 'test': 2}).fillna(series))
)
export_distribution['label_1_ratio'] = (
    export_distribution['label_1_count'] / export_distribution['total_rows']
).round(2)
export_distribution['label_0_ratio'] = (
    export_distribution['label_0_count'] / export_distribution['total_rows']
).round(2)

print('Export summary:')
display(export_summary_df)

print('export_df first 5 rows:')
display(export_df.head())

print('Final distribution by fold and dataset:')
display(export_distribution)

print('Columns kept:', list(export_columns.values()))

Export summary:


,output_path,rows_written,fold_count,datasets
0,C:\Users\纯情小火鸡\Desktop\bilby2\bilby2\Bilby cod...,4174,4,"test, train, valid"


export_df first 5 rows:


,fold_id,dataset,date,root_url,source_url,title,summary,title_normalized,title_en,type,label,title_lang
0,0,test,2025-10-31,www.mofcom.gov.cn,https://www.mofcom.gov.cn/bnjg/art/2025/art_13...,王新副局长主持召开相关猪肉及猪副产品反倾销案听证会,2025年10月31日，商务部召开猪肉及猪副产品反倾销案听证会，共有37家利害关系方参加，包...,王新副局长主持召开相关猪肉及猪副产品反倾销案听证会,Deputy Director Wang presided over the hearing...,bilby,0,zh
1,0,test,2025-10-31,www.mofcom.gov.cn,https://www.mofcom.gov.cn/gdtb/gzdt/art/2025/a...,昆明特办在云南召开企业培训会,2025年10月30日，昆明特办与云南省商务厅联合举办企业警示教育及业务培训会。商务部有关司...,昆明特办在云南召开企业培训会,Kunming Special Office held a corporate traini...,bilby,0,zh
2,0,test,2025-10-31,www.mofcom.gov.cn,https://www.mofcom.gov.cn/gdtb/gzdt/art/2025/a...,福州特办党支部赴宁德开展联合主题党日活动,10月30日，福州特办党支部书记刘福学率特派员与宁德市商务局、霞浦县商务局党员干部“三级联动...,福州特办党支部赴宁德开展联合主题党日活动,The Fuzhou Special Branch Party Branch went to...,bilby,0,zh
3,0,test,2025-10-31,www.mofcom.gov.cn,https://www.mofcom.gov.cn/gdtb/gzdt/art/2025/a...,刘福学特派员带队赴宁德市蕉城区和霞浦县调研,刘福学特派员10月29日至30日前往宁德市蕉城区和霞浦县进行调研，与当地商务主管部门和外贸企...,刘福学特派员带队赴宁德市蕉城区和霞浦县调研,Special Commissioner Liu Fuxue led a team to i...,bilby,0,zh
4,0,test,2025-10-31,www.mofcom.gov.cn,https://www.mofcom.gov.cn/xwfb/ldrhd/art/2025/...,习近平在亚太经合组织第三十二次领导人非正式会议第一阶段会议上的讲话（全文）,习近平在亚太经合组织会议上提出共建开放亚太经济的五点建议，包括维护多边贸易体制、营造开放经济...,习近平在亚太经合组织第三十二次领导人非正式会议第一阶段会议上的讲话全文,Xi Jinping's speech at the first stage meeting...,bilby,0,zh


Final distribution by fold and dataset:


,fold_id,dataset,total_rows,label_1_count,label_0_count,title_zh_count,title_en_count,label_1_ratio,label_0_ratio
1,0,train,465,40,425,465,0,0.09,0.91
2,0,valid,178,19,159,178,0,0.11,0.89
0,0,test,168,15,153,167,1,0.09,0.91
4,1,train,643,59,584,643,0,0.09,0.91
5,1,valid,168,15,153,167,1,0.09,0.91
3,1,test,161,8,153,160,1,0.05,0.95
7,2,train,813,74,739,812,1,0.09,0.91
8,2,valid,178,8,170,177,1,0.04,0.96
6,2,test,145,13,132,145,0,0.09,0.91
10,3,train,1025,83,942,1023,2,0.08,0.92


Columns kept: ['fold_id', 'dataset', 'date', 'root_url', 'source_url', 'title', 'summary', 'title_normalized', 'title_en', 'type', 'label']
